In [14]:
from openai import OpenAI
import json
import os

client = OpenAI()


In [3]:
# Example dummy function hard coded to return the same weather# In production, this could be your backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "72", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "72", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})



In [4]:

# Step 1: send the conversation and available functions to the model
messages = [{"role": "user", "content": "What's the weather like in San Francisco, Tokyo, and Paris?"}]
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]


In [5]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools,
    tool_choice="auto",  # auto is default, but we'll be explicit
)
response_message = response.choices[0].message


In [9]:
print(response_message)

ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_XmWJk6VoqLcY2HQXYaUuLfIa', function=Function(arguments='{"location": "San Francisco, CA"}', name='get_current_weather'), type='function'), ChatCompletionMessageToolCall(id='call_HY5ernwACrF0of9cSmrvGVx5', function=Function(arguments='{"location": "Tokyo, Japan"}', name='get_current_weather'), type='function'), ChatCompletionMessageToolCall(id='call_IEctQJRkGDnBPWclPWljALVP', function=Function(arguments='{"location": "Paris, France"}', name='get_current_weather'), type='function')])


In [10]:

tool_calls = response_message.tool_calls
# Step 2: check if the model wanted to call a function


In [11]:


# Step 3: call the function
# Note: the JSON response may not always be valid; be sure to handle errors
available_functions = {
    "get_current_weather": get_current_weather,
}  # only one function in this example, but you can have multiple
messages.append(response_message)  # extend conversation with assistant's reply


In [12]:
print(messages)

[{'role': 'user', 'content': "What's the weather like in San Francisco, Tokyo, and Paris?"}, ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_XmWJk6VoqLcY2HQXYaUuLfIa', function=Function(arguments='{"location": "San Francisco, CA"}', name='get_current_weather'), type='function'), ChatCompletionMessageToolCall(id='call_HY5ernwACrF0of9cSmrvGVx5', function=Function(arguments='{"location": "Tokyo, Japan"}', name='get_current_weather'), type='function'), ChatCompletionMessageToolCall(id='call_IEctQJRkGDnBPWclPWljALVP', function=Function(arguments='{"location": "Paris, France"}', name='get_current_weather'), type='function')])]


In [ ]:

# Step 4: send the info for each function call and function response to the model
for tool_call in tool_calls:
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(
        location=function_args.get("location"),
        unit=function_args.get("unit"),
    )
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )  # extend conversation with function response
second_response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
)  # get a new response from the model where it can see the function response



In [26]:
import ChatAPICallFunctionsLib as cfl
import AutoSTMFuctionsLib as stmfl
import inspect

In [29]:
tools_list = [cfl.chen_ming_algorithm,stmfl.load_txt_from_idl,stmfl.plot_topo]

In [36]:
tools = cfl.auto_jsontooldict(tools_list)

Error Occur Expecting value: line 1 column 1 (char 0)
Running again....
Error Occur Expecting value: line 1 column 1 (char 0)
Running again....


In [38]:
def get_current_weather(location, unit="fahrenheit"):
    a=1
def get_current_traffic(location):
    a=1

available_functions = {"get_current_weather": get_current_weather,"get_current_traffic": get_current_traffic}



In [51]:
functions_dict_path = 'toolsjsdict_example.json'

with open(functions_dict_path, 'r') as file:
    tools = json.load(file)

In [53]:
response = client.chat.completions.create(
    model="gpt-4-0613",
    messages=[{"role": "user", "content":"what is the weather in san francisco, also could you give me the traffic?"}],
    temperature=0,
    functions = tools,
    function_call = "auto",
    )

response_message = response.choices[0].message.function_call
print(response_message)

FunctionCall(arguments='{\n  "location": "San Francisco, CA"\n}', name='get_current_weather')
